### Build a transformer 

config => rope => expert => moe => attention => transformer block => transformer 


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange, repeat, reduce
from dataclasses import dataclass

# -- 1. Model Configuration -- #

@dataclass
class ModelArgs:
    dim: int = 512
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int | None = None # For Grouped Query Attention, if needed
    vocab_size: int = 32000
    # MoE specific
    n_experts: int = 4
    n_experts_per_tok: int = 2
    # RoPE specific
    rope_theta: float = 10000.0

# -- 2. RoPE Implementation -- #

def precompute_theta_pos_frequencies(head_dim: int, seq_len: int, theta: float):
    # As per the RoPE paper, the frequencies are related to theta^(2k/d)
    # where k is the dimension index and d is the head dimension
    assert head_dim % 2 == 0, "head_dim must be even"
    theta_numerator = torch.arange(0, head_dim, 2).float()
    theta = 1.0 / (theta ** (theta_numerator / head_dim))
    m = torch.arange(seq_len)
    freqs = torch.outer(m, theta).float()
    # We can think of this as complex numbers polar(r, theta)
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex

def apply_rotary_embeddings(x: torch.Tensor, freqs_complex: torch.Tensor):
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_complex = freqs_complex.unsqueeze(1) # Add head dimension
    x_rotated = x_complex * freqs_complex
    x_out = torch.view_as_real(x_rotated)
    x_out = x_out.reshape(*x.shape)
    return x_out.type_as(x)

# -- 3. MoE Implementation -- #

class Expert(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.w1 = nn.Linear(args.dim, args.dim * 2, bias=False)
        self.w2 = nn.Linear(args.dim * 2, args.dim, bias=False)
        self.w3 = nn.Linear(args.dim, args.dim * 2, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MOE(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.gate = nn.Linear(args.dim, args.n_experts, bias=False)
        self.experts = nn.ModuleList([Expert(args) for _ in range(args.n_experts)])
        self.n_experts_per_tok = args.n_experts_per_tok

    def forward(self, x):
        batch_size, seq_len, dim = x.shape
    
        # 1. Aplatir : de (bs, seq, dim) à (bs * seq, dim)
        x_flattened = x.view(-1, dim) 
        
        # 2. Ton code de routing actuel (exemple simplifié)
        router_logits = self.gate(x_flattened)
        routing_weights = F.softmax(router_logits, dim=-1)
        
        # Prépare un tenseur de sortie vide
        final_output = torch.zeros_like(x_flattened)
        
        # 3. La boucle sur les experts (C'est ici que l'erreur arrivait)
        for i, expert in enumerate(self.experts):
            # expert_mask doit être calculé sur x_flattened
            expert_mask = (torch.argmax(routing_weights, dim=-1) == i)
            
            if expert_mask.any():
                # Maintenant, les indices de expert_mask correspondront 
                # exactement aux lignes de x_flattened
                expert_input = x_flattened[expert_mask] 
                expert_output = expert(expert_input)
                
                # On remet le résultat au bon endroit
                final_output[expert_mask] = expert_output * routing_weights[expert_mask, i:i+1]
        # 4. Redonner la forme d'origine (bs, seq, dim)
        return final_output.view(batch_size, seq_len, dim)

# -- 4. Attention & Transformer Block -- #

class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads

        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)

    def forward(self, x, freqs_complex):
        batch_size, seq_len, _ = x.shape

        q = self.wq(x).view(batch_size, seq_len, self.n_heads, self.head_dim)
        k = self.wk(x).view(batch_size, seq_len, self.n_heads, self.head_dim)
        v = self.wv(x).view(batch_size, seq_len, self.n_heads, self.head_dim)

        # Apply RoPE
        q = apply_rotary_embeddings(q, freqs_complex)
        k = apply_rotary_embeddings(k, freqs_complex)

        q = q.transpose(1, 2) # (bs, n_heads, seq_len, head_dim)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(self.head_dim)
        # Apply causal mask
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device)
        scores = scores.masked_fill(mask, float('-inf'))
        attention_weights = F.softmax(scores, dim=-1)

        output = torch.matmul(attention_weights, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.wo(output)

class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.attention = Attention(args)
        self.feed_forward = MOE(args)
        self.attention_norm = nn.LayerNorm(args.dim)
        self.ffn_norm = nn.LayerNorm(args.dim)

    def forward(self, x, freqs_complex):
        # Pre-normalization and residual connection for attention
        h = x + self.attention(self.attention_norm(x), freqs_complex)
        # Pre-normalization and residual connection for FFN
        out = h + self.feed_forward(self.ffn_norm(h))
        return out

# -- 5. The Full Transformer Model -- #

class Transformer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        self.layers = nn.ModuleList([TransformerBlock(args) for _ in range(args.n_layers)])
        self.norm = nn.LayerNorm(args.dim)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

        # Precompute RoPE frequencies
        self.freqs_complex = precompute_theta_pos_frequencies(self.args.dim // self.args.n_heads, 2048, self.args.rope_theta) # Max seq len 2048

    def forward(self, tokens):
        batch_size, seq_len = tokens.shape
        h = self.tok_embeddings(tokens)
        
        # Ensure freqs_complex is on the same device as the input
        self.freqs_complex = self.freqs_complex.to(h.device)
        freqs = self.freqs_complex[:seq_len]

        for layer in self.layers:
            h = layer(h, freqs)
        
        h = self.norm(h)
        # We only care about the output of the last token for generation
        output = self.output(h[:, -1, :])
        return output

# -- 6. Example Usage -- #

if __name__ == '__main__':
    args = ModelArgs(dim=256, n_layers=4, n_heads=4, vocab_size=1000)
    model = Transformer(args)

    # Tie weights (optional but good practice)
    model.tok_embeddings.weight = model.output.weight

    print(model)

    # Create a dummy input
    dummy_input = torch.randint(0, 1000, (2, 100)) # Batch size 2, sequence length 100
    
    # Get model output
    logits = model(dummy_input)
    print(f"Output logits shape: {logits.shape}") # Should be (batch_size, vocab_size)


### Baby GPT 

tokeniser

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange, repeat, reduce
from dataclasses import dataclass

@dataclass
class ModelArgs: 
    head_dim: int = 32 
    n_head: int = 4
    dim: int = 128
    n_layers: int = 4        # Ajoute bien le : int
    vocab_size: int = 65     # Ajoute le : int
    max_seq_len: int = 256   # Ajoute le : int
    batch_size: int = 64     # Ajoute le : int
    lr: float = 1e-3
    dropout: float = 0.1    # Pour éviter que le modèle n'apprenne Shakespeare "par coe
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

 # Create vocabulary and mappings
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Convert entire dataset to a giant PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

# 90% Training, 10% Validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Vocab size: {vocab_size}")
print(f"Train size: {len(train_data)} tokens")
print(f"Val size: {len(val_data)} tokens")

def get_batch(split, args):

    # On choisit quel dataset utiliser
    data_source = train_data if split == 'train' else val_data
    
    # On génère des indices de départ au hasard (Batch Size indices)
    # On s'arrête avant la fin pour avoir assez de place pour max_seq_len + 1
    ix = torch.randint(len(data_source) - args.max_seq_len, (args.batch_size,))
    
    # On extrait les séquences d'entrée (x)
    x = torch.stack([data_source[i:i+args.max_seq_len] for i in ix])
    
    # On extrait les cibles (y) : c'est x décalé d'un vers la droite (+1)
    y = torch.stack([data_source[i+1:i+args.max_seq_len+1] for i in ix])
    
    return x, y
# Testons la fonction avec tes ModelArgs
x, y = get_batch('train', ModelArgs())
print(f"Forme de x: {x.shape}") # Devrait être (64, 256) selon ta config
print(f"Forme de y: {y.shape}")



class Attention(nn.Module):


    def __init__(self, ModelArgs:dim):
        super().__init__()
        self.n_heads = args.n_head
        self.head_dim = args.dim // args.n_head


        self.wq = nn.Linear(args.dim, head_dim * n_head, bias=False)
        self.wk = nn.Linear(args.dim, head_dim * n_head, bias=False)
        self.wv = nn.Linear(args.dim, head_dim * n_head, bias=False)
        self.wo = nn.Linear(head_dim * n_head, args.dim, bias=False)

    def forward(self, x):
        batch_size,seq_len, _ = x.shape
        # 1. Projections et split des têtes
        # On passe de (Batch, Seq_len, 128) -> (Batch, Seq_len, n_head(4), 32)
        q = self.wq(x).view(batch_size, seq_len, self.n_heads, self.head_dim)
        k = self.wk(x).view(batch_size, seq_len, self.n_heads, self.head_dim)
        v = self.wv(x).view(batch_size, seq_len, self.n_heads, self.head_dim)

        # 2. Transpose pour isoler les têtes : (Batch, n_head(4), Seq_len, 32)
        # C'est indispensable pour que le matmul se fasse sur les deux dernières dimensions (Seq_len, 32)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2) 

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # On crée un masque triangulaire de 0 et 1, on le transforme en booléen
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        # On remplit les 'True' (le futur) par -infini pour que le softmax les ignore
        scores = scores.masked_fill(mask, float('-inf'))
        # 5. Softmax + Multiplication par V
        weights = F.softmax(scores, dim=-1) # (B, 4, S, S)
        output = weights @ v               # (B, 4, S, 32)
        # 6. Recombiner les têtes (Concate) et projection finale
        # On remet la Séquence en 2ème : (B, S, 4, 32) -> puis View (B, S, 128)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.wo(output)


class Mlp(nn.Module):
    def __init__(self, ModelArgs: dim, attention = None):
        super().__init__()

        self.hidden1 = nn.Linear(args.dim, args.dim*2 , bias = True)
        self.hidden2 = nn.Linear(args.dim*2, args.dim , bias = True)
    
    def forward(self, x ): 
        return self.hidden2(F.relu(self.hidden1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, args): 
        super().__init__()
        self.attentionNorm = nn.LayerNorm(args.dim)
        self.attention = attention(args)
        self.ff = Mlp(args)
        self.ffNorm = nn.LayerNorm(args.dim) 

    def forward(self, x): 
        h = x + self.attention(self.attentionNorm(x))
        out  = h + self.ff(self.ffNorm(h))
    return out 

class GptModule(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.tokenEmbeddings = nn.Embeddings(args.vocab, args.dim)
        self.position_embedding_table = nn.Embeddings(args.max_seq_len, args.dim)
        self.transformerBlock = nn.ModuleList([TransformerBlock(args) for _ in range (args.n_layers)])
        self.ln_f = nn.LayerNorm(args)
        self.lm_head = nn.Linear(args.dim, args.vocab_size)

        
    def forward(self, token, targets=None):
        # B = Batch (ex: 64), T = Temps/Séquence (ex: 256)
        B, T = token.shape
        # 1. Transformation des lettres en vecteurs (B, T, C)
        tok_emb = self.tokenEmbeddings(token)
        # 2. Création des positions [0, 1, 2... T-1] et transformation en vecteurs (T, C)
        # On s'assure que les positions sont sur la même puce (CPU/GPU) que les tokens
        pos = torch.arange(T, device=token.device)
        pos_emb = self.position_embedding_table(pos)
        # 3. Fusion des infos : le modèle sait MAINTENANT quelle lettre est à quelle place
        # x devient (B, T, C) grâce au broadcasting
        x = tok_emb + pos_emb
        # 4. On passe dans la boucle de tous les blocs de Transformers
        for block in self.transformerBlock:
            x = block(x)
        # 5. Normalisation finale et prédiction des scores pour chaque lettre du vocabulaire
        x = self.ln_f(x)
        logits = self.lm_head(x) # Résultat : (B, T, vocab_size)
        # 6. Calcul de la perte (Optionnel, uniquement si on s'entraîne)
        loss = None
        if targets is not None:
            # PyTorch attend (N, C) pour la cross_entropy
            # N = toutes les lettres de tous les batchs (B*T)
            # C = les probabilités pour chaque lettre du vocabulaire
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss


Vocab size: 65
Train size: 1003854 tokens
Val size: 111540 tokens
Forme de x: torch.Size([64, 256])
Forme de y: torch.Size([64, 256])


In [ ]:
#import des composants 
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange, repeat, reduce
from dataclasses import dataclass
#définitions de la classe/config de notre architecture
@dataClass
class Config:
    dim:int = 512
    n_layers:int = 4
    n_heads:int = 4
    head_dim:int = 126 
    max_seq_len:int = 256 
    vocab_size:int = 65
    

#fonction pour position token 
# définition du mlp 
#definition de l'attention 
#définition transforme block 
#définitoon du module gpt